<a href="https://colab.research.google.com/github/sabinereis28/projeto-gestao-logexpress-2026/blob/main/otimizacao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
from scipy.optimize import milp, Bounds, LinearConstraint

# ----------------------------------------------------
# CENÁRIO BASE
# ----------------------------------------------------
# Maximização de: 300 * xA + 800 * xB
# Como o SciPy minimiza, invertemos os sinais dos coeficientes da função objetivo
c = -np.array([300, 800])

# Variáveis inteiras (xA, xB >= 0)
# Tipo: 1 representa variável inteira (Integer)
integrality = np.ones_like(c)

# Limites das variáveis: xA >= 0 e xB >= 0
bounds = Bounds([0, 0], [np.inf, np.inf])

# Restrições de recursos:
# 1. Horas: 2 * xA + 6 * xB <= 160
# 2. Combustível: 15 * xA + 50 * xB <= 1200
A_matrix = np.array([
    [2, 6],
    [15, 50]
])
ub_base = np.array([160, 1200])
lb_base = np.array([-np.inf, -np.inf])

constraints_base = LinearConstraint(A_matrix, lb_base, ub_base)

res_base = milp(c=c, bounds=bounds, constraints=constraints_base, integrality=integrality)

print("=== PROBLEMA 1: CENÁRIO BASE ===")
if res_base.success:
    xA_val = round(res_base.x[0])
    xB_val = round(res_base.x[1])
    lucro_base = -res_base.fun
    print(f"Status: Sucesso (Otimizado)")
    print(f"Fretes Locais (Tipo A): {xA_val}")
    print(f"Fretes Estaduais (Tipo B): {xB_val}")
    print(f"Lucro Máximo Obtido: R$ {lucro_base:,.2f}\n")
else:
    print("Status: Falha ao resolver o cenário base.\n")

# ----------------------------------------------------
# CENÁRIO WHAT-IF (Cenário de Crise: Escassez de Combustível)
# ----------------------------------------------------
# Nova restrição de combustível: 15 * xA + 50 * xB <= 500
ub_crise = np.array([160, 500])
constraints_crise = LinearConstraint(A_matrix, lb_base, ub_crise)

res_crise = milp(c=c, bounds=bounds, constraints=constraints_crise, integrality=integrality)

print("=== PROBLEMA 1: CENÁRIO WHAT-IF (CRISE DE COMBUSTÍVEL) ===")
if res_crise.success:
    xA_c_val = round(res_crise.x[0])
    xB_c_val = round(res_crise.x[1])
    lucro_crise = -res_crise.fun
    print(f"Status: Sucesso (Otimizado)")
    print(f"Fretes Locais (Tipo A): {xA_c_val}")
    print(f"Fretes Estaduais (Tipo B): {xB_c_val}")
    print(f"Lucro Reduzido na Crise: R$ {lucro_crise:,.2f}")
else:
    print("Status: Falha ao resolver o cenário de crise.")

=== PROBLEMA 1: CENÁRIO BASE ===
Status: Sucesso (Otimizado)
Fretes Locais (Tipo A): 80
Fretes Estaduais (Tipo B): 0
Lucro Máximo Obtido: R$ 24,000.00

=== PROBLEMA 1: CENÁRIO WHAT-IF (CRISE DE COMBUSTÍVEL) ===
Status: Sucesso (Otimizado)
Fretes Locais (Tipo A): 33
Fretes Estaduais (Tipo B): 0
Lucro Reduzido na Crise: R$ 9,900.00


In [2]:
import numpy as np
from scipy.optimize import milp, Bounds, LinearConstraint

# MINIMIZACAO DE CUSTOS DE TRANSPORTE
# Valores hipoteticos de custo por unidade (R$).
# Devem ser identificados como premissas do cenario simulado.

# Ordem das variaveis:
# SP_C1, SP_C2, SP_C3, RJ_C1, RJ_C2, RJ_C3

custos = np.array([100, 120, 180, 170, 150, 110])

# Todas as quantidades devem ser inteiras e nao negativas.
limites = Bounds(
    np.zeros(6),
    np.full(6, np.inf)
)

inteiras = np.ones(6)

# Restricoes:
# SP: no maximo 50 unidades
# RJ: no maximo 40 unidades
# C1: receber exatamente 30
# C2: receber exatamente 35
# C3: receber exatamente 25

matriz = np.array([
    [1, 1, 1, 0, 0, 0],
    [0, 0, 0, 1, 1, 1],
    [1, 0, 0, 1, 0, 0],
    [0, 1, 0, 0, 1, 0],
    [0, 0, 1, 0, 0, 1]
])

limite_inferior = np.array([
    -np.inf, -np.inf, 30, 35, 25
])

limite_superior = np.array([
    50, 40, 30, 35, 25
])

restricoes = LinearConstraint(
    matriz,
    limite_inferior,
    limite_superior
)

resultado = milp(
    c=custos,
    integrality=inteiras,
    bounds=limites,
    constraints=restricoes
)

print("=== MINIMIZACAO DE CUSTOS ===")

if resultado.success:
    x = np.rint(resultado.x).astype(int)

    print("Status: Solucao otima encontrada")
    print(f"SP -> C1: {x[0]} unidades")
    print(f"SP -> C2: {x[1]} unidades")
    print(f"SP -> C3: {x[2]} unidades")
    print(f"RJ -> C1: {x[3]} unidades")
    print(f"RJ -> C2: {x[4]} unidades")
    print(f"RJ -> C3: {x[5]} unidades")
    print(f"Custo minimo total: R$ {resultado.fun:,.2f}")

    print("\nATENCAO: custos unitarios hipoteticos,")
    print("utilizados para demonstracao do modelo.")
else:
    print("Nao foi possivel encontrar uma solucao.")

=== MINIMIZACAO DE CUSTOS ===
Status: Solucao otima encontrada
SP -> C1: 30 unidades
SP -> C2: 20 unidades
SP -> C3: 0 unidades
RJ -> C1: 0 unidades
RJ -> C2: 15 unidades
RJ -> C3: 25 unidades
Custo minimo total: R$ 10,400.00

ATENCAO: custos unitarios hipoteticos,
utilizados para demonstracao do modelo.
